# Stage 9c: Dynamic selection with no upper cap

**Question.** The frozen rule keeps every candidate scoring at least -3.898 but clamps to 3-15 items. What happens if the
upper limit of 15 is removed (floor of 3 kept, no maximum beyond the 50-candidate pool)?

**Method.** One retriever (nomic embeddings + BM25 + RRF) and one MiniLM reranker, each loaded once. For each of the 40
questions in `evals/handbook_eval.jsonl`, the top 50 fused candidates are scored by MiniLM once and cached; every rule
below is evaluated from that cache. Rules: uncapped cutoffs -3.68, -3.8 and -3.898; fixed top 10 and top 15; and the
frozen system (-3.898, clamped 3-15) as a reference.

**Caveats.** This is a plain evaluation at three given cutoffs; nothing is tuned here. It is **in-sample**: the same 40
questions were used to choose -3.898. Token counts use `handbook_qa.tokens.count_tokens` (cl100k), which only
approximates Claude's tokenizer. No Claude API calls are made.

In [1]:
from dataclasses import replace

import numpy as np
import pandas as pd

from handbook_qa import evaluation as ev
from handbook_qa.config import ROOT, load_settings
from handbook_qa.retrieval import DynamicSelection, Reranker, Retriever
from handbook_qa.tokens import count_tokens

pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 80)

# Load settings, retriever and reranker
N_CAND = 50
settings = load_settings()
r = settings.retrieval
retriever = Retriever.from_settings(replace(settings, retrieval=replace(r, rerank=False)))  # embedder loaded once
reranker = Reranker(r.rerank_model, settings.embed.model_cache, settings.embed.threads)        # MiniLM loaded once
questions = ev.load_eval_set(ROOT / "evals" / "handbook_eval.jsonl")
print("answer-key problems:", ev.resolve(questions, retriever.items) or "none")
# Load the frozen rule and check it
frozen = DynamicSelection.load(ROOT / "evals" / "frozen_dynamic_selection.json")
print(frozen)
assert frozen.cutoff == -3.898 and (frozen.min_keep, frozen.max_keep, frozen.candidates) == (3, 15, 50)

C:\Users\shaba\Documents\extra project\paradigm_assessment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


answer-key problems: none
DynamicSelection(cutoff=-3.898, candidates=50, min_keep=3, max_keep=15, model='Xenova/ms-marco-MiniLM-L-6-v2')


## Score the 50 candidates per question, once

In [2]:
# Score the top 50 candidates per question
rows = []
for q in questions:
    hits = retriever.search(q.question, k=N_CAND, rerank=False)   # top 50 fused
    assert len(hits) == N_CAND
    scores = reranker.score(q.question, [h.item.text for h in hits])
    for h, s in zip(hits, scores):
        rows.append({"id": q.id, "fused rank": h.fused_rank, "item": h.item.id, "score": float(s),
                     "tokens": count_tokens(h.item.text)})
# Per-question arrays of scores, recall, tokens
cache = pd.DataFrame(rows)
by_id = {q.id: q for q in questions}
IDS = [q.id for q in questions]
ranked = {i: g.sort_values("score", ascending=False, kind="stable") for i, g in cache.groupby("id", sort=False)}
SCORES = np.stack([ranked[i]["score"].to_numpy() for i in IDS])
RECALL = np.array([[ev.chunk_recall(ranked[i]["item"].tolist(), by_id[i], n) for n in range(N_CAND + 1)] for i in IDS])
TOKENS = np.stack([np.concatenate([[0], np.cumsum(ranked[i]["tokens"].to_numpy())]) for i in IDS])
ROWS = np.arange(len(IDS))
print(len(cache), "cached scores")

2000 cached scores


## Rules

Each rule only decides **n**, how many of the reranked candidates to keep. Recall is `chunk_recall(ids_kept, q, len(ids_kept))`
(the cache holds it for every n). Dynamic rules: count of scores >= cutoff, floor 3, and a cap of 50 (no cap) or 15 (frozen).

In [3]:
# Rule definitions
def dyn(cutoff, cap):
    return np.clip((SCORES >= cutoff).sum(1), 3, cap)

# Rules to compare
RULES = {
    "dynamic -3.68, no cap": dyn(-3.68, 50),
    "dynamic -3.8, no cap": dyn(-3.8, 50),
    "dynamic -3.898, no cap": dyn(-3.898, 50),
    "fixed top 10": np.full(40, 10),
    "fixed top 15": np.full(40, 15),
    "frozen: -3.898, clamp 3-15 (reference)": dyn(-3.898, 15),
}
# the frozen reference must agree with the retriever's own rule
assert all(frozen.count(list(SCORES[k])) == RULES["frozen: -3.898, clamp 3-15 (reference)"][k] for k in range(40))

# sanity checks
r10 = RECALL[ROWS, RULES["fixed top 10"]].mean()
rf = RECALL[ROWS, RULES["frozen: -3.898, clamp 3-15 (reference)"]].mean()
mf = RULES["frozen: -3.898, clamp 3-15 (reference)"].mean()
print(r10, rf, mf)
assert round(r10, 3) == 0.846, r10
assert round(rf, 3) == 0.825 and round(mf, 3) == 9.725, (rf, mf)

# Comparison table
table = []
for name, n in RULES.items():
    rec, tok = RECALL[ROWS, n], TOKENS[ROWS, n]
    table.append({"rule": name, "chunk recall": rec.mean(), "all chunks found": (rec == 1).mean(),
                  "mean chunks": n.mean(), "median chunks": np.median(n), "max chunks": n.max(),
                  "at floor (3)": int((n == 3).sum()), "mean tokens": tok.mean(), "max tokens": tok.max()})
comparison = pd.DataFrame(table).set_index("rule")
comparison.round(3)

0.8458333333333334 0.825 9.725


,chunk recall,all chunks found,mean chunks,median chunks,max chunks,at floor (3),mean tokens,max tokens
rule,,,,,,,,
"dynamic -3.68, no cap",0.862,0.725,13.700,9.5,43,10,7231.625,24998
"dynamic -3.8, no cap",0.862,0.725,14.125,10.5,44,9,7473.675,25627
"dynamic -3.898, no cap",0.871,0.750,14.500,11.0,45,8,7656.600,25950
fixed top 10,0.846,0.750,10.000,10.0,10,0,5327.825,6064
fixed top 15,0.846,0.750,15.000,15.0,15,0,7967.100,8878
"frozen: -3.898, clamp 3-15 (reference)",0.825,0.725,9.725,11.0,15,8,5119.625,8878


## Questions where an uncapped rule keeps more than 15, or its recall differs from top 10's

In [4]:
# Pick questions that differ from top 10
unc = [k for k in RULES if "no cap" in k]
t10 = RULES["fixed top 10"]
rec10 = RECALL[ROWS, t10]
sel = [i for i in ROWS if any(RULES[k][i] > 15 or RECALL[i, RULES[k][i]] != rec10[i] for k in unc)]
# Build per-question table rows
pq = []
for i in sel:
    row = {"id": IDS[i], "top-10 recall": rec10[i], "frozen n": RULES["frozen: -3.898, clamp 3-15 (reference)"][i]}
    for k in unc:
        row[k.replace("dynamic ", "") + " n"] = RULES[k][i]
        row[k.replace("dynamic ", "") + " recall"] = RECALL[i, RULES[k][i]]
    row["tokens (-3.898 no cap)"] = TOKENS[i, RULES["dynamic -3.898, no cap"][i]]
    pq.append(row)
per_question = pd.DataFrame(pq).set_index("id")
print(len(sel), "questions")
per_question.round(3)

18 questions


,top-10 recall,frozen n,"-3.68, no cap n","-3.68, no cap recall","-3.8, no cap n","-3.8, no cap recall","-3.898, no cap n","-3.898, no cap recall",tokens (-3.898 no cap)
id,,,,,,,,,
A-01,0.333,15,32,0.667,32,0.667,34,0.667,18110
A-04,1.000,4,3,0.667,3,0.667,4,1.000,1799
A-06,1.000,15,37,1.000,37,1.000,37,1.000,19134
A-08,1.000,15,22,1.000,23,1.000,24,1.000,12060
A-12,0.667,3,3,0.333,3,0.333,3,0.333,1409
B-01,1.000,15,15,1.000,16,1.000,17,1.000,7177
B-02,1.000,15,16,1.000,17,1.000,17,1.000,8335
B-09,1.000,15,21,1.000,23,1.000,23,1.000,12549
C1-04,0.000,15,27,1.000,27,1.000,28,1.000,15929


## Conclusion

Removing the cap raises mean chunk recall from 0.825 (frozen, clamped at 15) to 0.862-0.871, and slightly above fixed top 10 (0.846), but the gain is about one question's worth (the share of questions with every chunk found is 0.725-0.750 against 0.750 for top 10). It is bought with much larger contexts: -3.898 uncapped keeps 14.5 chunks on average and about 7,700 cl100k tokens (top 10: 5,300), and on some questions it keeps 40+ chunks and about 26,000 tokens, against a 6,000-token maximum for top 10. The extra recall comes from a few questions (for example C1-04 and C1-09) where the score cutoff reaches far down the list, while uncapped rules still lose chunks on others (A-12, C1-07) that top 10 finds. These are in-sample numbers on the 40 questions used to choose -3.898, with differences of a few hundredths, so they do not show that dropping the cap is better; fixed top 10 is the cheaper, more predictable option at near-equal recall.